# 课后练习解答（06.05_conv_bn_fusion）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 使用 fused.weight.data.copy_(conv.weight) 的原因是？
A. 避免对叶子 Parameter 做 in-place 操作触发 autograd 限制
B. 精度更高
C. 速度更快
D. 两者完全等价

**解答：** A

**解析：** 对叶子 Parameter 直接 copy_ 会报 in-place 错误，.data 绕过 autograd 追踪。


### 问题2（单选题）

**题目：** conv.bias 为 None 时，fused bias 应如何计算？
A. (0-running_mean)*scale+beta
B. running_mean*scale+beta
C. 0
D. beta

**解答：** A

**解析：** 把缺失 bias 视为 0，代入折叠公式即可。


### 问题3（单选题）

**题目：** padding_mode="reflect" 的 Conv 做 BN 折叠时，应？
A. 保持 padding_mode 不变
B. 改为 zeros
C. 报错
D. 去掉 padding

**解答：** A

**解析：** BN 折叠只改权重偏置，padding 语义必须原样保留。


### 问题4（多选题）

**题目：** fuse_model 递归实现必须？
A. 识别 Sequential 中 Conv+BN
B. 递归进入子模块
C. setattr 写回
D. 在 train 模式执行

**解答：** ABC

**解析：** 融合只在 eval 语义下等价，不能在 train 模式执行。


### 问题5（多选题）

**题目：** 融合正确性校验包括？
A. 输出最大误差
B. 输出 shape
C. 模块计数
D. 训练 loss 不变

**解答：** ABC

**解析：** 融合用于推理，训练 loss 不是校验目标。


### 问题6（判断题）

**题目：** torch.no_grad() 包裹权重写入可避免建立反向图。

**解答：** 对

**解析：** no_grad 下原地写叶子参数不会触发 in-place 限制。


### 问题7（判断题）

**题目：** 融合后的模型可以直接用于训练并保持与原始模型一致的梯度。

**解答：** 错

**解析：** 融合改变前向结构与统计量语义，只适用于推理/导出。


### 问题8（填空题）

**题目：** 折叠后新权重 W' = ____。

**解答：** gamma*W/sqrt(running_var+eps)


### 问题9（填空题）

**题目：** 折叠后新偏置 b' = ____。

**解答：** gamma*(b-running_mean)/sqrt(running_var+eps)+beta


### 问题10（简答题）

**题目：** 为什么用 Parameter.data 而不是直接修改 Parameter？

**解答：** Parameter 是带 autograd 追踪的叶子张量，直接 in-place 会触发限制；.data 不参与追踪，适合推理阶段确定性写值。


### 问题11（简答题）

**题目：** 为什么递归结果必须写回父模块？

**解答：** 递归会构造新的子模块对象，若不 setattr 写回，父模块仍持有旧模块引用，深层融合不会生效。


### 问题12（代码设计题）

**题目：** 完整实现 fuse_conv_bn_eval(conv, bn)，返回融合后的 Conv2d。

**解答：** ```python
def fuse_conv_bn_eval(conv, bn):
    fused = nn.Conv2d(
        conv.in_channels, conv.out_channels, conv.kernel_size,
        stride=conv.stride, padding=conv.padding, dilation=conv.dilation,
        groups=conv.groups, bias=True,
        padding_mode=conv.padding_mode, device=conv.weight.device,
    )
    scale = bn.weight / torch.sqrt(bn.running_var + bn.eps)
    with torch.no_grad():
        fused.weight.copy_(conv.weight * scale.view(-1, 1, 1, 1))
        bias = conv.bias if conv.bias is not None else torch.zeros_like(bn.bias)
        fused.bias.copy_((bias - bn.running_mean) * scale + bn.bias)
    return fused
```


### 问题13（单选题）

**题目：** running_var 中存在极小值且 eps 不足以稳定分母时，最稳健处理是？
A. 对 running_var 做 clamp 下界
B. 跳过该 BN
C. 使用 fp64
D. 删除权重

**解答：** A

**解析：** clamp 可保证分母有数值下界，避免权重爆炸；跳过或删除会破坏模型结构。


### 问题14（多选题）

**题目：** 融合后的 Conv2d 必须保持原卷积的？
A. groups
B. stride/padding/dilation
C. padding_mode
D. 输出通道数

**解答：** ABCD

**解析：** 几何与分组属性决定算子语义，任一不一致都会导致输出错误。


### 问题15（简答题）

**题目：** 设计融合正确性测试：输入规模、比较方法与容差，并说明为什么要分别测 eval 与 train？

**解答：** 使用固定随机输入如 (8,3,224,224)，在 model.eval() 下分别跑原始模型与融合模型，比较输出 shape 与最大绝对/相对误差，FP32 容差取 1e-4~1e-2；train 模式必须单独确认行为不等价，避免误用融合模型做训练。
